In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [2]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
output_file = DATA_DIR / "data_processed" / "flatfox_rentals_zentralschweiz.csv"
gpkg_path = DATA_DIR / "geodata" / "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"

# Wird von data_engineering_municipality.ipynb erzeugt.
municipality_file = DATA_DIR / "data_processed" / "gemeinde_indikatoren_zentralschweiz.csv"

CENTRAL_CANTONS = ["LU", "UR", "SZ", "OW", "NW", "ZG"]


In [3]:
def merge_flatfox_csvs(data_dir):
    """Liest alle Erhebungs-CSVs ein, entfernt identische Duplikate und behält die neueste Zeile je Inserat-ID."""
    
    data_dir = Path(data_dir)
    source_files = sorted((data_dir / "flatfox_data").glob("*/listings_all_fields.csv"))
    output_file = (data_dir / "data_processed" / "flatfox_listings_merged.csv")

    if not source_files:
        raise FileNotFoundError("Keine Erhebungsdateien gefunden.")

    frames = []
    report = []

    for source in source_files:
        frame = pd.read_csv(
            source,
            encoding="utf-8-sig",
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )

        frame["pk"] = frame["pk"].str.strip()

        if (frame["pk"].isna() | frame["pk"].eq("")).any():
            raise ValueError(f"Leere Inserat-ID in {source}")

        rows_read = len(frame)
        frame = frame.drop_duplicates()

        if frame["pk"].duplicated().any():
            raise ValueError(
                f"Widersprüchliche Zeilen mit gleicher pk in {source}"
            )

        report.append({
            "source_run": source.parent.name,
            "rows_read": rows_read,
            "unique_ids": len(frame),
            "identical_duplicates": rows_read - len(frame),
        })

        frame["source_run"] = source.parent.name
        frame["source_file"] = source.relative_to(data_dir).as_posix()
        frame["__run_at"] = pd.to_datetime(
            source.parent.name,
            format="%Y%m%dT%H%M%S_%fZ",
            utc=True,
        )
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = combined.sort_values("__run_at", kind="stable")

    if combined.empty:
        raise ValueError("Die Erhebungsdateien enthalten keine Inserate.")

    # Sichtungen über alle vorhandenen Erhebungen bestimmen.
    grouped = combined.groupby("pk")

    combined["first_seen_at"] = grouped["__run_at"].transform("min")
    combined["last_seen_at"] = grouped["__run_at"].transform("max")
    combined["scrape_count"] = grouped["source_run"].transform("nunique")

    # Je ID die gesamte neueste Zeile übernehmen.
    merged = (
        combined.drop_duplicates("pk", keep="last")
        .drop(columns="__run_at")
        .sort_values("pk")
        .reset_index(drop=True)
    )

    for column in ["first_seen_at", "last_seen_at"]:
        merged[column] = merged[column].dt.strftime(
            "%Y-%m-%dT%H:%M:%S.%fZ"
        )

    # Fertige Datei schreiben und danach das bisherige Ergebnis ersetzen.
    output_file.parent.mkdir(parents=True, exist_ok=True)
    temporary_file = output_file.with_suffix(".csv.tmp")
    merged.to_csv(temporary_file, index=False, encoding="utf-8-sig")
    temporary_file.replace(output_file)

    report_df = pd.DataFrame(report)

    print(f"Erhebungsdateien: {len(source_files)}")
    print(f"Eingelesene Zeilen: {report_df['rows_read'].sum()}")
    print(f"Eindeutige Inserate: {len(merged)}")
    print(f"Gespeichert: {output_file}")

    return merged, report_df

In [4]:
def filter_flatfox_rentals(df, gpkg_path):
    """
    Ordnet Inseraten anhand ihrer Koordinaten die Gemeinde und daraus
    den Kanton zu, dokumentiert uneindeutige oder fehlende Zuordnungen
    und grenzt auf die Zentralschweiz ein.
    """
    gpkg_path = Path(gpkg_path)

    if not gpkg_path.is_file():
        raise FileNotFoundError(
            f"GeoPackage-Datei nicht gefunden: {gpkg_path}"
        )

    df_rentals_geo = df.copy().reset_index(drop=True)
    layers = gpd.list_layers(gpkg_path)

    # Hilfsfunktion: Gewünschte Schweizer Gebietsflächen laden
    def load_areas(layer_keyword):
        layer_names = layers.loc[
            layers["name"].str.contains(
                layer_keyword, case=False, na=False
            ),
            "name",
        ].tolist()

        if len(layer_names) != 1:
            raise ValueError(
                f"Layer für '{layer_keyword}' nicht eindeutig gefunden: "
                f"{layers['name'].tolist()}"
            )

        areas = gpd.read_file(gpkg_path, layer=layer_names[0])

        areas = areas.rename(columns={
            column: column.upper()
            for column in areas.columns
            if column != areas.geometry.name
        })

        if areas.geometry.name != "geometry":
            areas = areas.rename_geometry("geometry")

        return areas.loc[areas["ICC"].eq("CH")].copy()

    # Inserat-Koordinaten vorbereiten
    longitude = pd.to_numeric(df_rentals_geo["longitude"], errors="coerce")
    latitude = pd.to_numeric(df_rentals_geo["latitude"], errors="coerce")

    valid_coordinates = (
        longitude.between(-180, 180)
        & latitude.between(-90, 90)
    ).fillna(False)

    points = gpd.GeoDataFrame(
        index=df_rentals_geo.index[valid_coordinates],
        geometry=gpd.points_from_xy(
            longitude.loc[valid_coordinates],
            latitude.loc[valid_coordinates],
        ),
        crs="EPSG:4326",
    )

    # Punkte eindeutig einem Gebiet zuordnen
    def assign_areas(areas, id_column):
        assigned = pd.Series(
            pd.NA, index=df_rentals_geo.index, dtype="object"
        )
        status = pd.Series(
            "koordinaten_ungueltig", index=df_rentals_geo.index
        )

        if valid_coordinates.any():
            matches = gpd.sjoin(
                points.to_crs(areas.crs),
                areas[[id_column, "geometry"]],
                how="left",
                predicate="intersects",
            )

            # Mehrere Geometrien derselben ID zählen als ein Gebiet.
            candidates = matches.groupby(level=0)[id_column].agg(
                lambda values: tuple(sorted(set(values.dropna())))
            )

            assigned = candidates.map(
                lambda values: values[0] if len(values) == 1 else pd.NA
            ).reindex(df_rentals_geo.index)

            status.loc[valid_coordinates] = "kein_treffer"

            ambiguous_indices = candidates[
                candidates.map(len).gt(1)
            ].index

            status.loc[ambiguous_indices] = "mehrdeutiger_treffer"
            status.loc[assigned.notna()] = "zugeordnet"

        return assigned, status

    # Gemeinden zuordnen
    municipalities = load_areas("hoheitsgebiet")

    # Gemeindegebiete auswählen, keine Seegebiete oder Kommunanzen.
    # Je nach Export liegt OBJEKTART als Zahl oder Bezeichnung vor.
    object_type = municipalities["OBJEKTART"]

    is_municipality = (
        pd.to_numeric(object_type, errors="coerce").eq(0)
        | object_type.astype("string")
            .str.strip()
            .str.lower()
            .eq("gemeindegebiet")
    ).fillna(False)

    municipalities = municipalities.loc[is_municipality].copy()

    if municipalities.empty:
        raise ValueError(
            "Keine Schweizer Gemeindegebiete gefunden. "
            "Bitte die Werte der Spalte OBJEKTART prüfen."
        )

    municipalities["gemeinde_bfs"] = pd.to_numeric(
        municipalities["BFS_NUMMER"], errors="coerce"
    ).astype("Int64")

    if municipalities["gemeinde_bfs"].isna().any():
        raise ValueError("Fehlende BFS-Gemeindenummer in den Geodaten.")

    canton_codes = dict(enumerate(
        [
            "ZH", "BE", "LU", "UR", "SZ", "OW", "NW", "GL", "ZG",
            "FR", "SO", "BS", "BL", "SH", "AR", "AI", "SG", "GR",
            "AG", "TG", "TI", "VD", "VS", "NE", "GE", "JU",
        ],
        start=1,
    ))

    municipalities["kanton_code"] = pd.to_numeric(
        municipalities["KANTONSNUMMER"], errors="coerce"
    ).map(canton_codes)

    if municipalities["kanton_code"].isna().any():
        raise ValueError("Unbekannte Kantonsnummer in den Geodaten.")

    municipality_ids, municipality_status = assign_areas(
        municipalities, "gemeinde_bfs"
    )

    df_rentals_geo["gemeinde_bfs"] = municipality_ids.astype("Int64")
    df_rentals_geo["gemeinde_geo_status"] = municipality_status

    # Name und Kanton direkt aus der zugeordneten Gemeinde ableiten
    municipality_info = (
        municipalities.drop_duplicates("gemeinde_bfs")
        .set_index("gemeinde_bfs")
    )

    df_rentals_geo["gemeinde_name"] = (
        df_rentals_geo["gemeinde_bfs"]
        .map(municipality_info["NAME"])
        .astype("string")
    )
    df_rentals_geo["kanton_geo"] = (
        df_rentals_geo["gemeinde_bfs"]
        .map(municipality_info["kanton_code"])
        .astype("string")
    )

    # Auf die Zentralschweiz eingrenzen
    df_central = df_rentals_geo.loc[
        df_rentals_geo["kanton_geo"].isin(CENTRAL_CANTONS)
    ].copy()

    return df_rentals_geo, df_central

In [5]:
def add_municipality_indicators(df_rentals, municipality_file):
    """Ergänzt die Gemeindeindikatoren (inkl. Einkommensmedian) anhand der
    BFS-Gemeindenummer und erhält alle Inserate.
    """
    indicators = pd.read_csv(municipality_file, encoding="utf-8-sig")

    # Name und Kanton stammen bereits aus der Geozuordnung der Inserate.
    indicators = indicators.drop(columns=["gemeinde_name", "kanton_geo"])
    indicators["gemeinde_bfs"] = indicators["gemeinde_bfs"].astype("Int64")

    rentals = df_rentals.copy()
    rentals["gemeinde_bfs"] = pd.to_numeric(
        rentals["gemeinde_bfs"], errors="raise"
    ).astype("Int64")

    return rentals.merge(
        indicators,
        on="gemeinde_bfs",
        how="left",
        validate="many_to_one",
        indicator="municipality_match_status",
        sort=False,
    )

In [6]:
# 1. Erhebungsdateien zusammenführen
df_merged, merge_report = merge_flatfox_csvs(DATA_DIR)

# Mietwohnungen und Häuser auswählen
mask = (df_merged["offer_type"].eq("RENT") & df_merged["object_category"].isin(["APARTMENT", "HOUSE"])).fillna(False)

df_filtered = df_merged.loc[mask].copy()

Erhebungsdateien: 12
Eingelesene Zeilen: 32112
Eindeutige Inserate: 3484
Gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data/data_processed/flatfox_listings_merged.csv


In [7]:
# 2. Kantone und Gemeinden zuordnen
df_rentals_geo, df_central = filter_flatfox_rentals(df_filtered, gpkg_path)

In [8]:
# 3. Gemeindeindikatoren ergänzen
df_central = add_municipality_indicators(df_central, municipality_file)

# 4. Für die Analyse verwendbare Inserate auswählen
analysis_mask = (
    # schliesst Inserate aus, welche Preis auf Anfrage oder keinen Monatspreis haben
    df_central["price_display"].notna()
    & df_central["price_unit"].eq("monthly")
    # schliesst fehlende Flächenangaben aus
    & df_central["surface_living"].notna()
    & df_central["street"].notna()
    & df_central["gemeinde_name"].notna()
    & df_central["is_furnished"].eq("False")
).fillna(False)

df_excluded = df_central.loc[~analysis_mask].copy()
df_central = df_central.loc[analysis_mask].reset_index(drop=True)

output_file.parent.mkdir(parents=True, exist_ok=True)
df_central.to_csv(output_file, index=False, encoding="utf-8-sig")

In [9]:
print(f"\nZusammengeführte Inserate: {len(df_merged)}")
print(f"Mietwohnungen und Häuser: {len(df_rentals_geo)}")
print(f"Davon Zentralschweiz und analysierbar: {len(df_central)}")
print(f"Ausgeschlossen (fehlende Angaben oder möbliert): {len(df_excluded)}")
print(f"CSV gespeichert: {output_file}")
print("\nStatus der Gemeindezuordnung:")

display(df_rentals_geo["gemeinde_geo_status"].value_counts())

print("\nInserate pro Zentralschweizer Kanton:")
display(df_central["kanton_geo"].value_counts().reindex(CENTRAL_CANTONS, fill_value=0))

print("\nStatus der Zuordnung der Gemeindeindikatoren:")
display(df_central["municipality_match_status"].value_counts())

print("\nGemeinden ohne Einkommenswert:")
display(df_central.loc[df_central["income_median_chf"].isna(), ["gemeinde_bfs", "gemeinde_name"],].drop_duplicates())

display(df_central)


Zusammengeführte Inserate: 3484
Mietwohnungen und Häuser: 1630
Davon Zentralschweiz und analysierbar: 513
Ausgeschlossen (fehlende Angaben oder möbliert): 394
CSV gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data/data_processed/flatfox_rentals_zentralschweiz.csv

Status der Gemeindezuordnung:


gemeinde_geo_status
zugeordnet    1630
Name: count, dtype: int64


Inserate pro Zentralschweizer Kanton:


kanton_geo
LU    358
UR     10
SZ     72
OW     13
NW      6
ZG     54
Name: count, dtype: Int64


Status der Zuordnung der Gemeindeindikatoren:


municipality_match_status
both          513
left_only       0
right_only      0
Name: count, dtype: int64


Gemeinden ohne Einkommenswert:


,gemeinde_bfs,gemeinde_name


,pk,slug,url,short_url,submit_url,status,created,offer_type,object_category,object_type,...,single_family_buildings_pct_year,foreign_population_pct_gemeindestand,foreign_population_pct_year,net_migration_per_1000_residents_gemeindestand,net_migration_per_1000_residents_year,employment_per_100_residents_year,employment_source_gemeindestand,employment_population_gemeindestand,employment_boundary_adjustment,municipality_match_status
0,1112177,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112177/,/1112177/,/en/listing/1112177/submit/,act,2024-02-19T12:41:36.786020+01:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
1,1112194,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112194/,/1112194/,/en/listing/1112194/submit/,act,2024-02-19T12:41:44.416739+01:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
2,1112211,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112211/,/1112211/,/en/listing/1112211/submit/,act,2024-02-19T12:41:51.887002+01:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
3,1112230,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112230/,/1112230/,/en/listing/1112230/submit/,act,2024-02-19T12:41:59.876138+01:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
4,1389528,papieri-ring-4-6330-cham,/en/flat/papieri-ring-4-6330-cham/1389528/,/1389528/,/en/listing/1389528/submit/,act,2024-11-12T17:48:49.797714+01:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,86479440,fannring-1-6403-kussnacht-am-rigi,/en/flat/fannring-1-6403-kussnacht-am-rigi/864...,/86479440/,/en/listing/86479440/submit/,act,2026-10-23T16:32:36.448133+02:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
509,86479615,tribschenstr-62b-6005-luzern,/en/flat/tribschenstr-62b-6005-luzern/86479615/,/86479615/,/en/listing/86479615/submit/,act,2026-10-23T17:24:19.627192+02:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
510,86479687,schachenstrasse-45-6010-kriens,/en/flat/schachenstrasse-45-6010-kriens/86479687/,/86479687/,/en/listing/86479687/submit/,act,2026-10-23T18:08:41.107880+02:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
511,86479713,althusstrasse-10-6182-escholzmatt,/en/flat/althusstrasse-10-6182-escholzmatt/864...,/86479713/,/en/listing/86479713/submit/,act,2026-10-23T18:32:14.164480+02:00,RENT,APARTMENT,APARTMENT,...,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen,both
